# MyWeather — Ứng dụng Thông Tin Thời Tiết  
**Môn:** Tư duy Tính toán  
**Sinh viên:** Lê Văn Quốc — MSSV: 24120421  
**Công cụ:** Python, Streamlit, OpenWeatherMap API  

## 1. Mục tiêu

Xây dựng ứng dụng **MyWeather** sử dụng Streamlit cho phép người dùng:
- Nhập tọa độ (latitude, longitude) lấy từ Google Maps
- Tự động xác định tên thành phố và quốc gia (ISO 3166-1 alpha-2) từ tọa độ
- Xem thông tin thời tiết hiện tại của thành phố đó
- Hiển thị bản đồ thời tiết với các lớp: nhiệt độ, gió, mây, mưa
- Xem chất lượng không khí (AQI)
- Xem biểu đồ dự báo thời tiết 5 ngày / 3 giờ

## 2. Phân tích Yêu cầu

### 2.1 Yêu cầu cơ bản
| Yêu cầu | API sử dụng | Endpoint |
|---|---|---|
| Tọa độ → tên thành phố + country code | Geocoding API | `/geo/1.0/reverse` |
| Thời tiết hiện tại | Current Weather API | `/data/2.5/weather` |
| Bản đồ thời tiết | Weather Maps 1.0 | `tile.openweathermap.org/map/{layer}` |
| Chất lượng không khí | Air Pollution API | `/data/2.5/air_pollution` |

### 2.2 Yêu cầu nâng cao
| Yêu cầu | API sử dụng | Endpoint |
|---|---|---|
| Biểu đồ nhiệt độ, gió, độ ẩm 5 ngày | 3-hourly Forecast API | `/data/2.5/forecast` |

### 2.3 Luồng xử lý cơ bản
```
Người dùng nhập tọa độ (Google Maps)
        ↓
Geocoding API → Tên thành phố + Country Code (ISO 3166-1 alpha-2)
        ↓
Current Weather API → Nhiệt độ, độ ẩm, gió, tình trạng
Air Pollution API   → AQI, PM2.5
Forecast API        → Dữ liệu 5 ngày / 3 giờ
Weather Map Tiles   → Bản đồ trực quan
        ↓
Streamlit UI hiển thị kết quả
```

## 3. Kiến trúc Tổng quát
 
### Sơ đồ Pipeline
 
```
User Input (lat, lon)
        |
        v
    app.py (Streamlit)
        |
        |-----> api/geocoding.py ---------> OpenWeatherMap Geocoding API
        |-----> api/weather.py -----------> OpenWeatherMap Current Weather API
        |-----> api/weather.py -----------> OpenWeatherMap Forecast API
        |-----> api/air_pollution.py -----> OpenWeatherMap Air Pollution API
        |-----> components/map_view.py ---> Weather Map Tiles
        |-----> components/charts.py -----> Plotly Charts
        |
        v
    Streamlit UI
```
 
### Cấu trúc thư mục
 
```
my-weather/
    api/
        __init__.py
        geocoding.py        (Reverse geocoding)
        weather.py          (Current weather + Forecast)
        air_pollution.py    (AQI)
    components/
        __init__.py
        map_view.py         (Folium weather map)
        charts.py           (Plotly forecast charts)
    .env                    (API key - khong commit)
    .gitignore
    app.py                  (Streamlit entry point)
    requirements.txt
```

## 4. Thiết lập API

### 4.1 Đăng ký OpenWeatherMap
1. Truy cập https://home.openweathermap.org/users/sign_in
2. Đăng ký tài khoản miễn phí
3. Vào **API Keys** → copy key
4. Lưu vào file `.env`:

```
OWM_API_KEY=your_key_here
```
### 4.2 Cac API su dung (Free tier)
 
| API | Giới hạn miễn phí |
|---|---|
| Geocoding API | 1,000 calls/ngày |
| Current Weather | 1,000 calls/ngày |
| 3-hour Forecast 5 days | 1,000 calls/ngày |
| Air Pollution | 1,000 calls/ngày |
| Weather Map Tiles | 1,000,000 tiles/ngày |
 
### 4.3 Test kết nối API

In [27]:
import requests
import os
from dotenv import load_dotenv
 
load_dotenv()
API_KEY = os.getenv("OWM_API_KEY")
 
# Test geocoding
lat, lon = 10.7769, 106.7009
r = requests.get(
    "https://api.openweathermap.org/geo/1.0/reverse",
    params={"lat": lat, "lon": lon, "limit": 1, "appid": API_KEY}
)
print("Status:", r.status_code)
print("Result:", r.json())

Status: 200
Result: [{'name': 'Ho Chi Minh City', 'local_names': {'cs': 'Ho Či Minovo Město', 'hr': 'Grad Ho Chi Minh', 'ru': 'Хошимин', 'et': 'Hồ Chí Minh', 'nl': 'Ho Chi Minhstad', 'eo': 'Ho-Chi-Minh-urbo', 'en': 'Ho Chi Minh City', 'eu': 'Ho Chi Minh Hiria', 'th': 'นครโฮจิมินห์', 'ja': 'ホーチミン市', 'sk': 'Hočiminovo Mesto', 'oc': 'Hô Chi Minh Vila', 'zh': '胡志明市', 'de': 'Ho-Chi-Minh-Stadt', 'el': 'Χο Τσι Μιν', 'km': 'ទីក្រុង\u200bហូ\u200bជី\u200bមិញ', 'fa': 'هوشی\u200cمین', 'sr': 'Хо Ши Мин', 'es': 'Ciudad Ho Chi Minh', 'ko': '호치민시', 'ms': 'Kota Ho Chi Minh', 'id': 'Kota Ho Chi Minh', 'lv': 'Hočimina', 'he': "הו צ'י מין סיטי", 'vi': 'Thành phố Hồ Chí Minh', 'fr': 'Hô Chi Minh-Ville', 'ar': 'مدينة هو تشي منه', 'ml': 'ഹോ ചി മിൻ നഗരം', 'it': 'Città di Ho Chi Minh'}, 'lat': 10.7763897, 'lon': 106.7011391, 'country': 'VN'}]


**Output mong doi:**
 
```
Status: 200
Result: [{'name': 'Ho Chi Minh City', 'country': 'VN', ...}]
```

## 5. Thiết kế Server / Module
 
### 5.1 api/geocoding.py

In [28]:
import requests
 
BASE = "https://api.openweathermap.org/geo/1.0/reverse"
 
def coords_to_city(lat: float, lon: float, api_key: str) -> dict:
    """Chuyen toa do thanh ten thanh pho va country code."""
    r = requests.get(BASE, params={
        "lat": lat, "lon": lon,
        "limit": 1, "appid": api_key
    })
    r.raise_for_status()
    data = r.json()
    if not data:
        return {}
    return data[0]  # tra ve: name, country (ISO 3166-1 alpha-2)

### 5.2 api/weather.py

In [29]:
import requests
 
CURRENT_URL = "https://api.openweathermap.org/data/2.5/weather"
FORECAST_URL = "https://api.openweathermap.org/data/2.5/forecast"
 
def get_current(city: str, country: str, api_key: str) -> dict:
    r = requests.get(CURRENT_URL, params={
        "q": f"{city},{country}",
        "appid": api_key,
        "units": "metric",
        "lang": "vi"
    })
    r.raise_for_status()
    return r.json()
 
def get_forecast(city: str, country: str, api_key: str) -> dict:
    r = requests.get(FORECAST_URL, params={
        "q": f"{city},{country}",
        "appid": api_key,
        "units": "metric",
        "lang": "vi"
    })
    r.raise_for_status()
    return r.json()

### 5.3 api/air_pollution.py

In [30]:
import requests
 
URL = "https://api.openweathermap.org/data/2.5/air_pollution"
AQI_LABEL = {1: "Tot", 2: "Kha", 3: "Trung binh", 4: "Xau", 5: "Rat xau"}
 
def get_aqi(lat: float, lon: float, api_key: str) -> dict:
    r = requests.get(URL, params={"lat": lat, "lon": lon, "appid": api_key})
    r.raise_for_status()
    data = r.json()
    aqi = data["list"][0]["main"]["aqi"]
    components = data["list"][0]["components"]
    return {"aqi": aqi, "label": AQI_LABEL[aqi], "components": components}

### 5.4 components/map_view.py

In [31]:
import folium
from streamlit_folium import st_folium
 
LAYER_OPTIONS = {
    "Nhiet do": "temp_new",
    "Gio": "wind_new",
    "May": "clouds_new",
    "Mua": "precipitation_new",
}
 
def render_weather_map(lat, lon, city, api_key, layer="temp_new"):
    m = folium.Map(location=[lat, lon], zoom_start=8)
    for label, code in LAYER_OPTIONS.items():
        folium.TileLayer(
            tiles=f"https://tile.openweathermap.org/map/{code}/{{z}}/{{x}}/{{y}}.png?appid={api_key}",
            attr="OpenWeatherMap",
            name=label,
            overlay=True,
            show=(code == layer),
        ).add_to(m)
    folium.LayerControl().add_to(m)
    folium.Marker([lat, lon], popup=city).add_to(m)
    st_folium(m, height=420, use_container_width=True)

### 5.5 components/charts.py

In [32]:
import plotly.express as px
import pandas as pd
import streamlit as st
 
def render_forecast_charts(forecast_json: dict):
    records = [
        {
            "Thoi gian": item["dt_txt"],
            "Nhiet do (C)": item["main"]["temp"],
            "Do am (%)": item["main"]["humidity"],
            "Gio (m/s)": item["wind"]["speed"],
        }
        for item in forecast_json["list"]
    ]
    df = pd.DataFrame(records)
 
    fig_temp = px.line(df, x="Thoi gian", y="Nhiet do (C)", title="Nhiet do 5 ngay")
    st.plotly_chart(fig_temp, use_container_width=True)
 
    fig_wind = px.line(df, x="Thoi gian", y="Gio (m/s)", title="Toc do gio 5 ngay")
    st.plotly_chart(fig_wind, use_container_width=True)
 
    fig_hum = px.bar(df, x="Thoi gian", y="Do am (%)", title="Do am 5 ngay")
    st.plotly_chart(fig_hum, use_container_width=True)

## 6. Demo từng Module

In [33]:
from api.geocoding import coords_to_city
from api.weather import get_current, get_forecast
from api.air_pollution import get_aqi
import os
from dotenv import load_dotenv
 
load_dotenv()
API_KEY = os.getenv("OWM_API_KEY")
 
# 1. Geocoding
geo = coords_to_city(10.7769, 106.7009, API_KEY)
print(f"Thanh pho: {geo['name']}, {geo['country']}")
 
# 2. Current weather
weather = get_current(geo['name'], geo['country'], API_KEY)
print(f"Nhiet do: {weather['main']['temp']}C")
print(f"Tinh trang: {weather['weather'][0]['description']}")
 
# 3. AQI
aqi = get_aqi(10.7769, 106.7009, API_KEY)
print(f"AQI: {aqi['aqi']} - {aqi['label']}")
print(f"PM2.5: {aqi['components']['pm2_5']} ug/m3")
 
# 4. Forecast
forecast = get_forecast(geo['name'], geo['country'], API_KEY)
print(f"So moc thoi gian: {len(forecast['list'])} (moi 3 gio)")
print(f"Moc dau tien: {forecast['list'][0]['dt_txt']} - {forecast['list'][0]['main']['temp']}C")

Thanh pho: Ho Chi Minh City, VN
Nhiet do: 30.1C
Tinh trang: mây đen u ám
AQI: 1 - Tốt
PM2.5: 5.59 ug/m3
So moc thoi gian: 40 (moi 3 gio)
Moc dau tien: 2026-05-19 15:00:00 - 29.75C


## 7. Chạy Ứng dụng
 
### Cài đặt dependencies
 
```bash
pip install -r requirements.txt
```
 
### Nội dung requirements.txt
 
```
streamlit>=1.35.0
requests>=2.31.0
python-dotenv>=1.0.0
folium>=0.16.0
streamlit-folium>=0.20.0
plotly>=5.22.0
pandas>=2.2.0
```
 
### Khởi động app
 
```bash
streamlit run app.py
```
 
App chạy tại: http://localhost:8501
 
### Hướng dẫn sử dụng
 
1. Mở Google Maps, click chuột phải vào địa điểm bất kỳ
2. Copy tọa đọ (số đầu = Latitude, số sau = Longitude)
3. Điền vào sidebar của app
4. Bấm "Tìm thông tin thời tiết"

## 8. Kiểm thử
 
| Test case | Tọa độ | Kết quả mong đợi | Kết quả thực tế |
|---|---|---|---|
| HCM City | 10.7769, 106.7009 | Ho Chi Minh City, VN | Đúng  |
| Hà Nội | 21.0285, 105.8542 | Hanoi, VN | Đúng  |
| Tokyo | 35.6762, 139.6503 | Tokyo, JP | Đúng  |
| Sydney | -33.8688, 151.2093 | Sydney, AU | Đúng  |    
| Giữa biển (0,0) | 0.0, 0.0 | Thông báo lỗi | Đúng  |
| Tọa độ không có data | 10.0, 100.0 | Thông báo lỗi | Đúng  |
 
**Nhận xét:** App xử lý đúng các trường hợp bình thường và edge case, không crash khi gặp tọa độ không hợp lệ.

## 9. Chức năng Mở rộng
 
### Đã triển khai
 
- **Biểu đồ dự báo 5 ngày**: nhiệt độ, tốc độ gió, độ ẩm theo từng 3 giờ (Plotly)
- **Weather map layers**: 4 lớp bạn có thể chọn
- **Error handling**: hiển thị thông báo lỗi thân thiện khi tọa độ không hợp lệ

### Có thể phát triển thêm
 
- Weather widgets (icon thời tiết động)
- Lưu danh sách thành phố yêu thích
- Quản lý tài khoản người dùng
- So sánh thời tiết nhiều thành phố